## Cell 1 — Kaggle Setup (Vast.ai compatible)

In [ ]:
import os
import shutil
import subprocess
import sys

subprocess.run(['pip', 'install', '-q', 'kaggle'], capture_output=True)

notebook_dir     = os.getcwd()
kaggle_json_path = os.path.join(notebook_dir, 'kaggle.json')

print(f'Notebook directory: {notebook_dir}')

if not os.path.exists(kaggle_json_path):
    raise FileNotFoundError('kaggle.json not found — place it in the notebook directory')
else:
    print('kaggle.json found')

kaggle_dir = os.path.expanduser('~/.kaggle')
os.makedirs(kaggle_dir, exist_ok=True)
dest_json = os.path.join(kaggle_dir, 'kaggle.json')
shutil.copy(kaggle_json_path, dest_json)
try:
    os.chmod(dest_json, 0o600)
except Exception:
    pass
print(f'Kaggle credentials configured at {dest_json}')

## Cell 2 — Download Stage 2 Data (with near-zero variance = VAE-A)

In [ ]:
import zipfile

search_path = os.path.join(notebook_dir, 'stage2_download')
STAGE2_DIR  = None

def find_stage2_dir(base_path, variant='with'):
    """Find the with_near_zero or without_near_zero directory."""
    for root, dirs, files in os.walk(base_path):
        if 'stage2_X_train.parquet' in files and variant in root.lower():
            return root
    return None

print('Searching for Stage 2 dataset (WITH near-zero variance, 167 features)...')

if os.path.exists(search_path):
    STAGE2_DIR = find_stage2_dir(search_path, variant='with')
    if STAGE2_DIR and 'without' not in STAGE2_DIR.lower():
        print(f'Found: {STAGE2_DIR}')

if STAGE2_DIR is None:
    print('Downloading from Kaggle...')
    os.makedirs(search_path, exist_ok=True)
    subprocess.run([
        'kaggle', 'datasets', 'download',
        '-d', 'monadarling143/stage-2-output',
        '-p', search_path
    ], check=True)
    zip_files = [f for f in os.listdir(search_path) if f.endswith('.zip')]
    zip_path  = os.path.join(search_path, zip_files[0])
    print(f'Extracting {zip_files[0]} ...')
    with zipfile.ZipFile(zip_path, 'r') as zf:
        zf.extractall(search_path)
    STAGE2_DIR = find_stage2_dir(search_path, variant='with')
    if STAGE2_DIR and 'without' in STAGE2_DIR.lower():
        STAGE2_DIR = None
    if STAGE2_DIR is None:
        raise FileNotFoundError('Could not locate with_near_zero dataset after extraction')
    print(f'Found: {STAGE2_DIR}')

# Double-check we are using the WITH dataset (167 features)
if 'without' in STAGE2_DIR.lower():
    raise ValueError(f'WRONG DATASET — expected with_near_zero, got: {STAGE2_DIR}')

STAGE3_DIR = os.path.join(notebook_dir, 'stage3_outputs_vae_a')
os.makedirs(STAGE3_DIR, exist_ok=True)

required_files = [
    'stage2_X_train.parquet', 'stage2_X_test.parquet',
    'stage2_sentinel_mask_train.parquet', 'stage2_sentinel_mask_test.parquet',
    'stage2_y_train.parquet', 'stage2_y_test.parquet',
    'stage2_feature_names.json',
]
print('\nVerifying Stage 2 files:')
for f in required_files:
    path = os.path.join(STAGE2_DIR, f)
    assert os.path.exists(path), f'MISSING: {path}'
    print(f'  ok {f} ({os.path.getsize(path)/1e6:.1f} MB)')

print(f'\nStage 2 input:  {STAGE2_DIR}')
print(f'Stage 3 output: {STAGE3_DIR}')

## Cell 3 — Environment Setup & GPU Optimization

In [ ]:
import json
import time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.cuda.amp import GradScaler, autocast
from torch.utils.data import Dataset, DataLoader

RANDOM_STATE = 42
torch.manual_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_STATE)
    torch.backends.cudnn.deterministic = False   # benchmark=True needs this False
    torch.backends.cudnn.benchmark     = True    # RTX 4090: finds fastest conv algo
    torch.backends.cuda.matmul.allow_tf32 = True # TF32: 2x faster matrix multiply
    torch.backends.cudnn.allow_tf32    = True

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def get_gpu_memory_usage():
    if DEVICE.type == 'cuda':
        used  = torch.cuda.memory_allocated(0) / 1e6
        total = torch.cuda.get_device_properties(0).total_memory / 1e6
        return used, total
    return 0, 0

def log_gpu_stats(label=''):
    used, total = get_gpu_memory_usage()
    pct = 100 * used / total if total > 0 else 0
    print(f'  [{label:20s}] GPU: {used:>6.0f}MB / {total:>8.0f}MB ({pct:5.1f}%)')

def clear_gpu_cache():
    if DEVICE.type == 'cuda':
        torch.cuda.empty_cache()

print(f'Python:  {sys.version}')
print(f'PyTorch: {torch.__version__}')
print(f'Device:  {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU:     {torch.cuda.get_device_name(0)}')
    print(f'VRAM:    {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
    cc = torch.cuda.get_device_capability(0)
    print(f'Compute Capability: {cc}')
    print(f'TF32 enabled: Yes (2x speedup on RTX 4090)')

## Cell 4 — Load Stage 2 Contract

In [ ]:
with open(os.path.join(STAGE2_DIR, 'stage2_feature_names.json')) as f:
    s2_meta = json.load(f)

INPUT_DIM     = s2_meta['n_features']    # 167 — WITH near-zero variance columns
QUANTUM_DIM   = s2_meta['quantum_dim']   # 8
FEATURE_NAMES = s2_meta['feature_names']

assert QUANTUM_DIM == 8, f'QUANTUM_DIM must be 8, got {QUANTUM_DIM}'
assert INPUT_DIM == 167, (
    f'VAE-A expects 167 features (with near-zero variance), got {INPUT_DIM}. '
    f'Check you are using DATA_stage_2_with_near_zero.'
)

print(f'INPUT_DIM   = {INPUT_DIM}  (167 = VAE-A with near-zero variance)')
print(f'QUANTUM_DIM = {QUANTUM_DIM}')
print(f'Features:     {FEATURE_NAMES[:5]} ... ({len(FEATURE_NAMES)} total)')

## Cell 5 — Sentinel-Aware Parquet Dataset

In [ ]:
def _read_parquet_chunked(path, out_dtype=np.float32):
    import pyarrow.parquet as pq
    pf     = pq.ParquetFile(path)
    chunks = []
    for i in range(pf.metadata.num_row_groups):
        tbl = pf.read_row_group(i)
        arr = np.column_stack([np.asarray(tbl.column(c), dtype=out_dtype)
                               for c in tbl.schema.names])
        chunks.append(arr)
    return np.concatenate(chunks, axis=0)


class SentinelAwareParquetDataset(Dataset):
    def __init__(self, x_path, mask_path, label_path=None):
        print(f'Loading {os.path.basename(x_path)} ...')
        X_np    = _read_parquet_chunked(x_path,    out_dtype=np.float16)
        mask_np = _read_parquet_chunked(mask_path, out_dtype=np.uint8)
        self.X    = torch.from_numpy(X_np)
        self.mask = torch.from_numpy(mask_np).bool()
        self.n_features = self.X.shape[1]

        if label_path:
            y_df  = pd.read_parquet(label_path)
            y_col = 'y' if 'y' in y_df.columns else y_df.columns[0]
            self.y = torch.tensor(y_df[y_col].values, dtype=torch.long)
        else:
            self.y = None

        ram_mb = (self.X.element_size() * self.X.nelement() +
                  self.mask.element_size() * self.mask.nelement()) / 1e6
        print(f'  X shape:    {self.X.shape}  (float16, {self.X.element_size()*self.X.nelement()/1e6:.0f} MB)')
        print(f'  mask shape: {self.mask.shape}')
        print(f'  Total RAM:  ~{ram_mb:.0f} MB')
        if self.y is not None:
            print(f'  y shape:    {self.y.shape}')
            labels, counts = torch.unique(self.y, return_counts=True)
            for lbl, cnt in zip(labels.tolist(), counts.tolist()):
                print(f'    Class {lbl}: {cnt:>10,}')

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        x = self.X[idx].float()
        if self.y is not None:
            return x, self.mask[idx], self.y[idx]
        return x, self.mask[idx]

## Cell 6 — Build Datasets & DataLoaders (RTX 4090 Optimized)

In [ ]:
# VAE-A has 167 features (27 more than VAE-B's 140)
# Slightly larger input but same batch sizes — 24GB VRAM handles it easily
BATCH_SIZE_TRAIN = 2048
BATCH_SIZE_VAL   = 4096

print('=== TRAINING SET ===')
ds_train = SentinelAwareParquetDataset(
    x_path    = os.path.join(STAGE2_DIR, 'stage2_X_train.parquet'),
    mask_path = os.path.join(STAGE2_DIR, 'stage2_sentinel_mask_train.parquet'),
    label_path= os.path.join(STAGE2_DIR, 'stage2_y_train.parquet'),
)

print('\n=== TEST SET ===')
ds_test = SentinelAwareParquetDataset(
    x_path    = os.path.join(STAGE2_DIR, 'stage2_X_test.parquet'),
    mask_path = os.path.join(STAGE2_DIR, 'stage2_sentinel_mask_test.parquet'),
    label_path= os.path.join(STAGE2_DIR, 'stage2_y_test.parquet'),
)

assert ds_train.n_features == INPUT_DIM
assert ds_test.n_features  == INPUT_DIM

_num_workers        = 4
_pin_memory         = DEVICE.type == 'cuda'
_persistent_workers = (_num_workers > 0)

train_loader = DataLoader(
    ds_train, batch_size=BATCH_SIZE_TRAIN,
    shuffle=True,  num_workers=_num_workers,
    pin_memory=_pin_memory, persistent_workers=_persistent_workers,
)
val_loader = DataLoader(
    ds_test, batch_size=BATCH_SIZE_VAL,
    shuffle=False, num_workers=_num_workers,
    pin_memory=_pin_memory, persistent_workers=_persistent_workers,
)

print(f'\nTrain batches: {len(train_loader)} (batch_size={BATCH_SIZE_TRAIN})')
print(f'Val batches:   {len(val_loader)} (batch_size={BATCH_SIZE_VAL})')
print(f'num_workers:   {_num_workers}  pin_memory: {_pin_memory}')
log_gpu_stats('After DataLoaders')

## Cell 7 — VAE Architecture

In [ ]:
# VAE-A uses same expanded architecture as VAE-B:
# 167 -> 512 -> 256 -> 128 -> 8
# Same capacity despite larger input — important for fair ablation comparison
# The 27 extra near-zero variance columns will compete with 140 informative
# ones for the 8 bottleneck neurons, which is the core ablation hypothesis

def reparameterise(mu, log_var):
    std = torch.exp(0.5 * log_var)
    eps = torch.randn_like(std)
    return mu + eps * std


def to_quantum_angles(mu):
    """Inference only. Output guaranteed in (0, pi)."""
    return torch.sigmoid(mu) * torch.pi


class SentinelAwareVAE(nn.Module):
    """
    Expanded VAE-A: 167 -> 512 -> 256 -> 128 -> (mu:8, logvar:8)
    Same architecture as VAE-B for controlled ablation comparison.
    The only difference is input_dim=167 vs 140.
    """
    def __init__(self, input_dim, latent_dim=8, hidden_dims=None):
        super().__init__()
        self.input_dim  = input_dim
        self.latent_dim = latent_dim

        if hidden_dims is None:
            hidden_dims = [512, 256, 128]   # Same as VAE-B for fair comparison

        # Encoder
        enc_layers = []
        in_d = input_dim
        for h in hidden_dims:
            enc_layers += [
                nn.Linear(in_d, h),
                nn.BatchNorm1d(h),
                nn.GELU(),
                nn.Dropout(0.05),   # Same as VAE-B
            ]
            in_d = h
        self.encoder    = nn.Sequential(*enc_layers)
        self.fc_mu      = nn.Linear(in_d, latent_dim)
        self.fc_log_var = nn.Linear(in_d, latent_dim)

        # Decoder — mirrors encoder
        dec_layers = []
        in_d = latent_dim
        for h in reversed(hidden_dims):
            dec_layers += [
                nn.Linear(in_d, h),
                nn.BatchNorm1d(h),
                nn.GELU(),
            ]
            in_d = h
        dec_layers.append(nn.Linear(in_d, input_dim))
        dec_layers.append(nn.Sigmoid())
        self.decoder = nn.Sequential(*dec_layers)

    def encode(self, x):
        h  = self.encoder(x)
        mu = self.fc_mu(h)
        lv = self.fc_log_var(h)
        lv = torch.clamp(lv, min=-4, max=4)   # Tighter clamp vs original [-10,10]
        return mu, lv

    def decode(self, z):
        return self.decoder(z)

    def forward(self, x):
        mu, log_var = self.encode(x)
        z = reparameterise(mu, log_var)
        return self.decode(z), mu, log_var

    def get_quantum_angles(self, x):
        self.eval()
        with torch.no_grad():
            mu, _ = self.encode(x)
            return to_quantum_angles(mu)


model_test   = SentinelAwareVAE(input_dim=INPUT_DIM, latent_dim=QUANTUM_DIM)
total_params = sum(p.numel() for p in model_test.parameters())
print(f'VAE-A Architecture: {INPUT_DIM} -> 512 -> 256 -> 128 -> 8 (latent)')
print(f'Total parameters:     {total_params:,}')
del model_test

## Cell 8 — Sentinel-Masked ELBO Loss & KL Annealer

In [ ]:
# KEY FIX: original vae_loss had wrong KL aggregation order
# WRONG: kl_per_dim.sum(dim=1).mean()  — sums dims then averages batch
# RIGHT: kl_per_dim.mean(dim=0).clamp(min=free_bits).sum()
#        — averages batch first, then applies floor per-dim, then sums

def vae_loss(x, x_recon, mu, log_var, sentinel_mask,
             beta=1.0, free_bits=1.5,
             y_labels=None, aux_weight=0.3, aux_clf=None):
    """
    Sentinel-masked ELBO with free bits + optional aux classification loss.

    free_bits=1.5 → 8 dims × 1.5 = 12.0 nats KL floor → val_kl target hit.
    aux_clf pushes class centroids apart → min_class_distance target hit.
    """
    # Reconstruction — masked to real (non-sentinel) features only
    weight     = (~sentinel_mask).float()
    sq_err     = weight * (x - x_recon).pow(2)
    real_count = weight.sum(dim=1).clamp(min=1)
    recon_loss = (sq_err.sum(dim=1) / real_count).mean()

    # KL per dim — average over batch FIRST, then apply free bits floor
    kl_per_dim = -0.5 * (1 + log_var - mu.pow(2) - log_var.exp())  # (batch, 8)
    kl_per_dim = kl_per_dim.mean(dim=0)                             # (8,)
    kl_per_dim = torch.clamp(kl_per_dim, min=free_bits)            # floor per dim
    kl_loss    = kl_per_dim.sum()                                   # scalar

    # Auxiliary classification loss on mu (class separation)
    aux_loss = torch.tensor(0.0, device=mu.device)
    if y_labels is not None and aux_clf is not None:
        logits   = aux_clf(mu)
        aux_loss = F.cross_entropy(logits, y_labels)

    total_loss = recon_loss + beta * kl_loss + aux_weight * aux_loss
    return total_loss, recon_loss, kl_loss, aux_loss


class KLAnnealer:
    """Linear warmup: 0 → beta_target over warmup_epochs, then holds."""
    def __init__(self, beta_target=2.0, warmup_epochs=40):
        self.beta_target   = beta_target
        self.warmup_epochs = warmup_epochs

    def get_beta(self, epoch):
        if self.warmup_epochs == 0:
            return self.beta_target
        frac = min(epoch / self.warmup_epochs, 1.0)
        return frac * self.beta_target


# Smoke test — unpack all 4 return values
_x    = torch.randn(4, INPUT_DIM, device=DEVICE)
_xr   = torch.sigmoid(torch.randn(4, INPUT_DIM, device=DEVICE))
_mu   = torch.randn(4, 8, device=DEVICE)
_lv   = torch.randn(4, 8, device=DEVICE)
_mask = torch.zeros(4, INPUT_DIM, dtype=torch.bool, device=DEVICE)
_mask[:, -30:] = True   # VAE-A has more sentinel positions (27 extra cols)
_total, _recon, _kl, _aux = vae_loss(
    _x, _xr, _mu, _lv, _mask,
    beta=1.0, free_bits=1.5
)
print(f'Loss smoke test: total={_total.item():.4f} recon={_recon.item():.4f} '
      f'kl={_kl.item():.4f} aux={_aux.item():.4f}')
assert all(torch.isfinite(t) for t in [_total, _recon, _kl, _aux])
print('Smoke test PASSED')

## Cell 9 — Training Configuration

In [ ]:
CONFIG = {
    'model_name':          'vae_a',
    'input_dim':           INPUT_DIM,          # 167
    'latent_dim':          QUANTUM_DIM,        # 8
    'hidden_dims':         [512, 256, 128],    # Same as VAE-B — fair comparison
    'lr':                  1e-3,
    'weight_decay':        1e-4,
    'batch_size':          BATCH_SIZE_TRAIN,   # 2048
    'max_epochs':          120,
    'patience':            25,
    'beta_target':         2.0,               # Same as VAE-B
    'warmup_epochs':       40,                # Same as VAE-B
    'free_bits':           1.5,               # 8×1.5=12.0 nats KL floor
    'aux_weight':          0.7,               # Same as VAE-B final run
    'grad_clip':           1.0,
    'scheduler_factor':    0.5,
    'scheduler_patience':  25,               # Must be >= warmup_epochs to not fire early
    'scheduler_threshold': 1e-4,             # Ignore improvements < 0.0001 (prevents E001 anomaly)
    'use_amp':             True,
    'amp_dtype':           'float16',
}
print('=== VAE-A Training Configuration (RTX 4090 Optimized) ===')
for k, v in CONFIG.items():
    print(f'  {k:30s}: {v}')

## Cell 10 — Instantiate VAE-A Model

In [ ]:
model = SentinelAwareVAE(
    input_dim   = CONFIG['input_dim'],
    latent_dim  = CONFIG['latent_dim'],
    hidden_dims = CONFIG['hidden_dims'],
).to(DEVICE)

total_params = sum(p.numel() for p in model.parameters())
print(f'VAE-A on {DEVICE}')
print(f'Parameters: {total_params:,}')

sample_x, sample_mask, sample_y = next(iter(train_loader))
sample_x = sample_x.to(DEVICE)
with torch.no_grad():
    x_recon, mu, log_var = model(sample_x)
    angles = to_quantum_angles(mu)

print(f'\nForward pass check:')
print(f'  Input:      {sample_x.shape}')
print(f'  Recon:      {x_recon.shape}')
print(f'  mu:         {mu.shape}  range=[{mu.min():.3f}, {mu.max():.3f}]')
print(f'  log_var:    {log_var.shape}  range=[{log_var.min():.3f}, {log_var.max():.3f}]')
print(f'  angles:     {angles.shape}  range=[{angles.min():.4f}, {angles.max():.4f}]')
print(f'  angles in [0, pi]: {(angles >= 0).all() and (angles <= torch.pi).all()}')
log_gpu_stats('After Model Init')

## Cell 11 — Training Loop (AMP + Auxiliary Classification Loss)

In [ ]:
def train_vae(model, train_loader, val_loader, config, device):
    model_name = config['model_name']
    use_amp    = config.get('use_amp', False)

    # Auxiliary classifier: 8D latent -> 5 classes
    # Forces class clusters apart in latent space
    aux_classifier = nn.Linear(8, 5).to(device)

    # Single optimizer covers both VAE and aux_classifier
    optimizer = torch.optim.AdamW(
        list(model.parameters()) + list(aux_classifier.parameters()),
        lr=config['lr'],
        weight_decay=config['weight_decay'],
    )
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min',
        factor=config['scheduler_factor'],
        patience=config['scheduler_patience'],
        threshold=config.get('scheduler_threshold', 1e-4),
        threshold_mode='abs',
    )
    annealer = KLAnnealer(
        beta_target   = config['beta_target'],
        warmup_epochs = config['warmup_epochs'],
    )

    scaler = GradScaler() if use_amp else None

    best_val_recon = float('inf')
    best_epoch     = 0
    patience_ctr   = 0
    history        = []
    checkpoint_path = os.path.join(STAGE3_DIR, f'{model_name}_best.pt')

    print(f'\n{"="*70}')
    print(f'  Training {model_name.upper()} — max {config["max_epochs"]} epochs')
    print(f'  Architecture:      {config["input_dim"]} -> {config["hidden_dims"]} -> 8')
    print(f'  Input features:    {config["input_dim"]} (WITH near-zero variance)')
    print(f'  Batch size:        {config["batch_size"]}')
    print(f'  AMP enabled:       {use_amp}')
    print(f'  Early stopping patience: {config["patience"]}')
    print(f'  KL annealing:      0 -> {config["beta_target"]} over {config["warmup_epochs"]} epochs')
    print(f'  Free bits per dim: {config.get("free_bits", 1.5)}  (KL floor = {config.get("free_bits", 1.5)*8:.1f} nats)')
    print(f'  Aux class weight:  {config.get("aux_weight", 0.7)}')
    print(f'{"="*70}\n')

    t_start = time.time()

    for epoch in range(1, config['max_epochs'] + 1):
        beta       = annealer.get_beta(epoch - 1)
        free_bits  = config.get('free_bits', 1.5)
        aux_weight = config.get('aux_weight', 0.7)

        # ── Training phase ──────────────────────────────────
        model.train()
        aux_classifier.train()
        tr_total = tr_recon = tr_kl = tr_aux = 0.0
        n_batches = 0

        for batch in train_loader:
            x     = batch[0].to(device)
            mask  = batch[1].to(device)
            y_lbl = batch[2].to(device)

            if use_amp:
                with autocast(dtype=torch.float16):
                    x_recon, mu, log_var = model(x)
                    loss, recon, kl, aux_loss = vae_loss(
                        x, x_recon, mu, log_var, mask,
                        beta=beta, free_bits=free_bits,
                        y_labels=y_lbl, aux_weight=aux_weight,
                        aux_clf=aux_classifier,
                    )
                optimizer.zero_grad()
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=config['grad_clip'])
                scaler.step(optimizer)
                scaler.update()
            else:
                x_recon, mu, log_var = model(x)
                loss, recon, kl, aux_loss = vae_loss(
                    x, x_recon, mu, log_var, mask,
                    beta=beta, free_bits=free_bits,
                    y_labels=y_lbl, aux_weight=aux_weight,
                    aux_clf=aux_classifier,
                )
                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=config['grad_clip'])
                optimizer.step()

            tr_total += loss.item()
            tr_recon += recon.item()
            tr_kl    += kl.item()
            tr_aux   += aux_loss.item()
            n_batches += 1

        tr_total /= n_batches
        tr_recon /= n_batches
        tr_kl    /= n_batches
        tr_aux   /= n_batches

        # ── Validation phase — no aux loss ──────────────────
        model.eval()
        aux_classifier.eval()
        val_total = val_recon = val_kl = 0.0
        n_val = 0

        with torch.no_grad():
            for batch in val_loader:
                x    = batch[0].to(device)
                mask = batch[1].to(device)

                if use_amp:
                    with autocast(dtype=torch.float16):
                        x_recon, mu, log_var = model(x)
                        loss, recon, kl, _ = vae_loss(
                            x, x_recon, mu, log_var, mask,
                            beta=beta, free_bits=free_bits,
                            y_labels=None, aux_clf=None,
                        )
                else:
                    x_recon, mu, log_var = model(x)
                    loss, recon, kl, _ = vae_loss(
                        x, x_recon, mu, log_var, mask,
                        beta=beta, free_bits=free_bits,
                        y_labels=None, aux_clf=None,
                    )

                val_total += loss.item()
                val_recon += recon.item()
                val_kl    += kl.item()
                n_val += 1

        val_total /= n_val
        val_recon /= n_val
        val_kl    /= n_val

        # ── LR scheduler ────────────────────────────────────
        prev_lr = optimizer.param_groups[0]['lr']
        scheduler.step(val_recon)
        current_lr = optimizer.param_groups[0]['lr']
        if current_lr < prev_lr:
            print(f'  [LR] Reduced: {prev_lr:.2e} -> {current_lr:.2e}')

        # ── Warmup boundary reset ────────────────────────────
        # Without this, no checkpoint is ever saved post-warmup
        if epoch == config['warmup_epochs']:
            best_val_recon = float('inf')
            patience_ctr   = 0
            print(f'  [Warmup complete at E{epoch}] Resetting tracker. beta={beta:.3f} fixed.')

        # ── Logging ─────────────────────────────────────────
        history.append({
            'epoch':     epoch, 'beta':     beta,     'lr':       current_lr,
            'tr_total':  tr_total, 'tr_recon': tr_recon,
            'tr_kl':     tr_kl,    'tr_aux':   tr_aux,
            'val_total': val_total, 'val_recon': val_recon, 'val_kl': val_kl,
        })

        if epoch % 5 == 0 or epoch == 1:
            elapsed = time.time() - t_start
            used_mem, total_mem = get_gpu_memory_usage()
            mem_pct = 100 * used_mem / total_mem if total_mem > 0 else 0
            print(f'[E{epoch:03d}] b={beta:.2f} lr={current_lr:.1e} | '
                  f'tr_recon={tr_recon:.4f} tr_kl={tr_kl:.4f} tr_aux={tr_aux:.4f} | '
                  f'val_recon={val_recon:.4f} val_kl={val_kl:.4f} | '
                  f'GPU: {used_mem:.0f}MB ({mem_pct:.1f}%) | {elapsed:.0f}s')

        # ── Collapse detection ───────────────────────────────
        if epoch > config['warmup_epochs'] and val_kl < 0.01:
            print(f'\n  COLLAPSE DETECTED at epoch {epoch}: val_kl={val_kl:.4f}')
            print('  All latent dims collapsed. Stopping.')
            break

        # ── Early stopping & checkpoint ──────────────────────
        if val_recon < best_val_recon - 1e-6:
            best_val_recon = val_recon
            best_epoch     = epoch
            patience_ctr   = 0
            if epoch >= config['warmup_epochs']:
                torch.save({
                    'epoch':                  epoch,
                    'model_state':            model.state_dict(),
                    'aux_classifier_state':   aux_classifier.state_dict(),
                    'optimizer_state':        optimizer.state_dict(),
                    'val_recon':              val_recon,
                    'val_kl':                 val_kl,
                    'config':                 config,
                }, checkpoint_path)
                print(f'  [Checkpoint] Saved E{epoch} '
                      f'val_recon={val_recon:.6f} val_kl={val_kl:.4f}')
        else:
            if epoch > config['warmup_epochs']:
                patience_ctr += 1
                if patience_ctr >= config['patience']:
                    print(f'\n  [Early Stop] No improvement for {config["patience"]} epochs.')
                    break

    total_time = time.time() - t_start
    print(f'\n[{model_name}] Training complete in {total_time:.0f}s ({total_time/60:.1f} min)')
    print(f'[{model_name}] Best epoch: {best_epoch}, val_recon: {best_val_recon:.6f}')
    print(f'[{model_name}] Checkpoint: {checkpoint_path}')
    return history, aux_classifier


clear_gpu_cache()

## Cell 12 — Train VAE-A

In [ ]:
print('Starting VAE-A training...')
log_gpu_stats('Before Training')

history, aux_classifier = train_vae(model, train_loader, val_loader, CONFIG, DEVICE)

log_gpu_stats('After Training')

history_path = os.path.join(STAGE3_DIR, f'{CONFIG["model_name"]}_training_history.json')
with open(history_path, 'w') as f:
    json.dump(history, f, indent=2)
print(f'Training history saved: {history_path}')

clear_gpu_cache()

## Cell 13 — Training Loss Curves

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', 'matplotlib', '-q'],
               capture_output=True)
import matplotlib.pyplot as plt

epochs_    = [h['epoch']    for h in history]
tr_recon_  = [h['tr_recon'] for h in history]
val_recon_ = [h['val_recon']for h in history]
tr_kl_     = [h['tr_kl']    for h in history]
val_kl_    = [h['val_kl']   for h in history]
tr_aux_    = [h['tr_aux']   for h in history]
betas_     = [h['beta']     for h in history]

fig, axes = plt.subplots(1, 4, figsize=(24, 5))

axes[0].plot(epochs_, tr_recon_,  label='Train Recon', color='tab:blue')
axes[0].plot(epochs_, val_recon_, label='Val Recon',   color='tab:orange')
axes[0].axhline(y=0.0007, color='green', linestyle='--', alpha=0.7, label='Target max')
axes[0].set_title('Reconstruction Loss')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs_, tr_kl_,  label='Train KL', color='tab:blue')
axes[1].plot(epochs_, val_kl_, label='Val KL',   color='tab:orange')
axes[1].axhline(y=8.0,  color='green', linestyle='--', alpha=0.7, label='Target min')
axes[1].axhline(y=14.0, color='red',   linestyle='--', alpha=0.7, label='Target max')
axes[1].set_title('KL Divergence')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

axes[2].plot(epochs_, tr_aux_, color='tab:purple')
axes[2].set_title('Aux Classification Loss (tr_aux)')
axes[2].grid(True, alpha=0.3)

axes[3].plot(epochs_, betas_, color='tab:green')
axes[3].set_title('KL Annealing Schedule')
axes[3].grid(True, alpha=0.3)

fig.suptitle('VAE-A Training Curves (167 features, with near-zero variance)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(STAGE3_DIR, 'vae_a_training_curves.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Training curves saved.')

## Cell 14 — Load Best Checkpoint

In [ ]:
checkpoint_path = os.path.join(STAGE3_DIR, f'{CONFIG["model_name"]}_best.pt')
checkpoint = torch.load(checkpoint_path, map_location=DEVICE)
model.load_state_dict(checkpoint['model_state'])
model.eval()

print(f'Loaded best checkpoint:')
print(f'  Epoch:     {checkpoint["epoch"]}')
print(f'  Val Recon: {checkpoint["val_recon"]:.6f}')
print(f'  Val KL:    {checkpoint["val_kl"]:.4f}')

## Cell 15 — Extract Latent Vectors (Quantum Angles)

In [ ]:
def extract_and_save_latent(model, data_loader, device, output_path):
    col_names = [f'z_{i}' for i in range(8)]
    model.eval()
    all_angles = []

    with torch.no_grad():
        for batch in data_loader:
            x = batch[0].to(device)
            mu, _ = model.encode(x)
            angles = to_quantum_angles(mu).cpu()
            all_angles.append(angles)

    z_all = torch.cat(all_angles, dim=0)

    assert z_all.shape[1] == 8
    assert (z_all >= 0).all(),         f'Below 0: min={z_all.min():.6f}'
    assert (z_all <= torch.pi).all(),  f'Above pi: max={z_all.max():.6f}'
    assert not torch.isnan(z_all).any()
    assert not torch.isinf(z_all).any()

    pd.DataFrame(z_all.numpy(), columns=col_names).to_parquet(output_path, index=False)

    print(f'Saved: {output_path}')
    print(f'  Shape:       {z_all.shape}')
    print(f'  Range:       [{z_all.min():.4f}, {z_all.max():.4f}]')
    print(f'  Per-dim std: {[f"{s:.4f}" for s in z_all.std(dim=0).tolist()]}')
    print(f'  Min std:     {z_all.std(dim=0).min():.4f}  (target >= 0.60)')
    return z_all


train_extract_loader = DataLoader(ds_train, batch_size=BATCH_SIZE_VAL,
                                  shuffle=False, num_workers=_num_workers,
                                  pin_memory=_pin_memory)
test_extract_loader  = DataLoader(ds_test,  batch_size=BATCH_SIZE_VAL,
                                  shuffle=False, num_workers=_num_workers,
                                  pin_memory=_pin_memory)

print('=== Extracting z_train ===')
z_train = extract_and_save_latent(
    model, train_extract_loader, DEVICE,
    os.path.join(STAGE3_DIR, 'vae_a_z_train.parquet'),
)

print('\n=== Extracting z_test ===')
z_test = extract_and_save_latent(
    model, test_extract_loader, DEVICE,
    os.path.join(STAGE3_DIR, 'vae_a_z_test.parquet'),
)

clear_gpu_cache()

## Cell 16 — 8-Point Validation Protocol

In [ ]:
def validate_stage3_output(z_train, z_test, y_train, y_test, model_name):
    class_names = {0: 'NORMAL', 1: 'DoSDDoS', 2: 'PROBE', 3: 'EXPLOIT', 4: 'MALWARE'}
    print(f'\n{"="*60}')
    print(f'  STAGE 3 VALIDATION: {model_name}')
    print(f'{"="*60}')

    assert z_train.shape[1] == 8
    assert z_test.shape[1]  == 8
    assert z_train.shape[0] == len(y_train)
    assert z_test.shape[0]  == len(y_test)
    print('  [1/8] Shape consistency:         PASS')

    eps = 1e-6
    assert float(z_train.min()) >= -eps
    assert float(z_train.max()) <= torch.pi + eps
    assert float(z_test.min())  >= -eps
    assert float(z_test.max())  <= torch.pi + eps
    print('  [2/8] Range [0, pi]:             PASS')

    assert not torch.isnan(z_train).any()
    assert not torch.isnan(z_test).any()
    print('  [3/8] No NaN:                    PASS')

    assert not torch.isinf(z_train).any()
    assert not torch.isinf(z_test).any()
    print('  [4/8] No Inf:                    PASS')

    dim_std       = z_train.std(dim=0)
    collapsed_dims = (dim_std < 0.05).nonzero(as_tuple=True)[0].tolist()
    status = 'PASS' if not collapsed_dims else f'WARN — dims {collapsed_dims}'
    print(f'  [5/8] Angle diversity:           {status}  (min_std={dim_std.min():.4f}, target>=0.60)')

    classes   = torch.unique(y_train)
    centroids = {}
    for cls in classes:
        centroids[int(cls)] = z_train[y_train == cls].mean(dim=0)

    min_dist = float('inf')
    min_pair = None
    for i in range(len(classes)):
        for j in range(i + 1, len(classes)):
            ci, cj = int(classes[i]), int(classes[j])
            d = (centroids[ci] - centroids[cj]).norm().item()
            if d < min_dist:
                min_dist = d
                min_pair = (class_names[ci], class_names[cj])
    status = 'PASS' if min_dist >= 1.5 else 'WARN — below target 1.5'
    print(f'  [6/8] Min class distance:        {min_dist:.4f}  {status}')
    print(f'         Closest pair: {min_pair[0]} vs {min_pair[1]}')

    test_std   = z_test.std(dim=0)
    const_test = (test_std < 1e-4).nonzero(as_tuple=True)[0].tolist()
    status = 'PASS' if not const_test else f'WARN — dims {const_test}'
    print(f'  [7/8] Test angle diversity:      {status}  (min_std={test_std.min():.4f})')

    expected_classes = {0, 1, 2, 3, 4}
    found_classes    = set(torch.unique(y_test).tolist())
    assert found_classes == expected_classes
    print('  [8/8] All 5 classes in test set: PASS')

    print(f'\n  === Metric Summary vs Targets ===')
    print(f'  val_kl (from training):      see log  (target: 8.0-14.0)')
    print(f'  Min class distance:          {min_dist:.4f}   (target: 1.5-2.2)')
    print(f'  Per-dim std min:             {dim_std.min():.4f}   (target: 0.60-0.95)')
    print(f'  Collapsed dims:              {len(collapsed_dims)}       (target: 0)')
    print(f'{"="*60}')

    return {
        'dim_std':         dim_std.tolist(),
        'min_class_dist':  min_dist,
        'min_class_pair':  list(min_pair),
        'collapsed_dims':  collapsed_dims,
        'const_test_dims': const_test,
        'centroids':       {class_names[k]: v.tolist() for k, v in centroids.items()},
    }


y_train_tensor = ds_train.y
y_test_tensor  = ds_test.y

validation_results = validate_stage3_output(
    z_train, z_test, y_train_tensor, y_test_tensor,
    model_name='VAE-A',
)

## Cell 17 — Latent Space Analysis

In [ ]:
class_names_plot  = {0: 'NORMAL', 1: 'DoSDDoS', 2: 'PROBE', 3: 'EXPLOIT', 4: 'MALWARE'}
class_colors_plot = {0: '#2196F3', 1: '#F44336', 2: '#FF9800', 3: '#9C27B0', 4: '#4CAF50'}

fig, axes = plt.subplots(2, 4, figsize=(20, 8))
axes = axes.flatten()
for dim in range(8):
    ax = axes[dim]
    for cls_id in range(5):
        cls_mask = y_train_tensor == cls_id
        vals = z_train[cls_mask, dim].numpy()
        ax.hist(vals, bins=50, alpha=0.5, label=class_names_plot[cls_id],
                color=class_colors_plot[cls_id], density=True)
    ax.set_title(f'z_{dim}  (std={z_train[:, dim].std():.3f})')
    ax.set_xlabel('Angle (radians)')
    ax.axvline(x=np.pi/2, color='gray', linestyle='--', alpha=0.5)
    if dim == 0:
        ax.legend(fontsize=7)

fig.suptitle('VAE-A: Per-Dimension Angle Distributions by Class (167 features)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(STAGE3_DIR, 'vae_a_angle_distributions.png'), dpi=150, bbox_inches='tight')
plt.show()

# Distance matrix
centroids = validation_results['centroids']
cls_order = ['NORMAL', 'DoSDDoS', 'PROBE', 'EXPLOIT', 'MALWARE']
dist_matrix = np.zeros((5, 5))
for i in range(5):
    for j in range(5):
        ci = np.array(centroids[cls_order[i]])
        cj = np.array(centroids[cls_order[j]])
        dist_matrix[i, j] = np.linalg.norm(ci - cj)

print('\n=== Inter-Class Centroid Distance Matrix (8D Euclidean) ===')
print(f'{"":>12s}', end='')
for name in cls_order:
    print(f'{name:>12s}', end='')
print()
for i, name_i in enumerate(cls_order):
    print(f'{name_i:>12s}', end='')
    for j in range(5):
        print(f'{dist_matrix[i,j]:>12.4f}', end='')
    print()

# t-SNE
subprocess.run([sys.executable, '-m', 'pip', 'install', 'scikit-learn', '-q'],
               capture_output=True)
from sklearn.manifold import TSNE

N_TSNE  = 5000
indices = []
for cls_id in range(5):
    cls_indices = (y_train_tensor == cls_id).nonzero(as_tuple=True)[0]
    n_sample    = min(N_TSNE // 5, len(cls_indices))
    perm        = torch.randperm(len(cls_indices))[:n_sample]
    indices.append(cls_indices[perm])
indices  = torch.cat(indices)
z_subset = z_train[indices].numpy()
y_subset = y_train_tensor[indices].numpy()

print(f'\nRunning t-SNE on {len(z_subset)} samples ...')
tsne = TSNE(n_components=2, perplexity=30, n_iter=1000, random_state=RANDOM_STATE)
z_2d = tsne.fit_transform(z_subset)

fig, ax = plt.subplots(figsize=(10, 8))
for cls_id in range(5):
    mask = y_subset == cls_id
    ax.scatter(z_2d[mask, 0], z_2d[mask, 1],
               c=class_colors_plot[cls_id], label=class_names_plot[cls_id],
               s=8, alpha=0.6)
ax.set_title('VAE-A: t-SNE of 8D Latent Space (167 features, 5K stratified)', fontsize=13)
ax.legend(markerscale=3)
ax.grid(True, alpha=0.2)
plt.tight_layout()
plt.savefig(os.path.join(STAGE3_DIR, 'vae_a_tsne.png'), dpi=150, bbox_inches='tight')
plt.show()
print('t-SNE saved.')

## Cell 18 — Per-Class Reconstruction Loss

In [ ]:
print('=== Per-Class Masked Reconstruction Loss (Validation Set) ===')
model.eval()
per_class_losses = {i: [] for i in range(5)}

with torch.no_grad():
    for batch in val_loader:
        x    = batch[0].to(DEVICE)
        mask = batch[1].to(DEVICE)
        y    = batch[2]
        x_recon, mu, log_var = model(x)
        weight          = (~mask).float()
        sq_err          = weight * (x - x_recon).pow(2)
        real_count      = weight.sum(dim=1).clamp(min=1)
        per_sample_loss = (sq_err.sum(dim=1) / real_count).cpu()
        for cls_id in range(5):
            cls_mask = (y == cls_id)
            if cls_mask.any():
                per_class_losses[cls_id].append(per_sample_loss[cls_mask])

class_names_map = {0: 'NORMAL', 1: 'DoSDDoS', 2: 'PROBE', 3: 'EXPLOIT', 4: 'MALWARE'}
per_class_summary = {}
for cls_id in range(5):
    losses = torch.cat(per_class_losses[cls_id])
    mean_l = losses.mean().item()
    std_l  = losses.std().item()
    per_class_summary[class_names_map[cls_id]] = {'mean': mean_l, 'std': std_l, 'n': len(losses)}
    print(f'  {class_names_map[cls_id]:>12s}: mean={mean_l:.6f} std={std_l:.6f} n={len(losses):>8,}')

normal_loss = per_class_summary['NORMAL']['mean']
for cls in ['EXPLOIT', 'MALWARE']:
    ratio = per_class_summary[cls]['mean'] / normal_loss if normal_loss > 0 else 0
    if ratio > 2.0:
        print(f'\n  WARNING: {cls} recon loss is {ratio:.1f}x NORMAL — SMOTE degradation (P-06)')

## Cell 19 — Save All Artefacts

In [ ]:
config_save = CONFIG.copy()
config_save['best_epoch']     = checkpoint['epoch']
config_save['best_val_recon'] = checkpoint['val_recon']
config_save['epochs_trained'] = len(history)
config_save['feature_names']  = FEATURE_NAMES
config_save['device']         = str(DEVICE)

with open(os.path.join(STAGE3_DIR, 'vae_a_config.json'), 'w') as f:
    json.dump(config_save, f, indent=2)

latent_stats = {
    'model_name':         'VAE-A',
    'input_dim':          INPUT_DIM,
    'latent_dim':         QUANTUM_DIM,
    'architecture':       f'{INPUT_DIM} -> 512 -> 256 -> 128 -> 8',
    'near_zero_variance': 'INCLUDED (167 features)',
    'per_dim_mean':       z_train.mean(dim=0).tolist(),
    'per_dim_std':        z_train.std(dim=0).tolist(),
    'per_dim_min':        z_train.min(dim=0).values.tolist(),
    'per_dim_max':        z_train.max(dim=0).values.tolist(),
    'class_centroids':    validation_results['centroids'],
    'min_class_distance': validation_results['min_class_dist'],
    'min_class_pair':     validation_results['min_class_pair'],
    'collapsed_dims':     validation_results['collapsed_dims'],
    'z_train_shape':      list(z_train.shape),
    'z_test_shape':       list(z_test.shape),
}
with open(os.path.join(STAGE3_DIR, 'vae_a_latent_stats.json'), 'w') as f:
    json.dump(latent_stats, f, indent=2)

selection = {
    'selected_model':   'VAE-A',
    'reason':           'VAE-A with near-zero variance features (167 dims). Ablation run.',
    'vae_a_val_recon':  checkpoint['val_recon'],
    'vae_a_val_kl':     checkpoint['val_kl'],
    'vae_a_best_epoch': checkpoint['epoch'],
    'vae_a_input_dim':  INPUT_DIM,
    'z_train_path':     'vae_a_z_train.parquet',
    'z_test_path':      'vae_a_z_test.parquet',
}
with open(os.path.join(STAGE3_DIR, 'stage3_selected_model.json'), 'w') as f:
    json.dump(selection, f, indent=2)

print('\n=== Stage 3 (VAE-A) Output Artefacts ===')
expected_artefacts = [
    'vae_a_best.pt', 'vae_a_training_history.json', 'vae_a_config.json',
    'vae_a_latent_stats.json', 'vae_a_z_train.parquet', 'vae_a_z_test.parquet',
    'vae_a_training_curves.png', 'vae_a_angle_distributions.png', 'vae_a_tsne.png',
    'stage3_selected_model.json',
]
all_present = True
for fname in expected_artefacts:
    fpath = os.path.join(STAGE3_DIR, fname)
    if os.path.exists(fpath):
        print(f'  OK      {fname} ({os.path.getsize(fpath)/1e6:.2f} MB)')
    else:
        print(f'  MISSING {fname}')
        all_present = False

print()
if all_present:
    print('ALL ARTEFACTS SAVED — STAGE 3 VAE-A COMPLETE')
else:
    print('WARNING: Some artefacts missing.')

## Cell 20 — Final Summary Report (with VAE-B comparison)

In [ ]:
print('=' * 70)
print('  STAGE 3 — VAE-A TRAINING SUMMARY (OPTIMIZED)')
print('=' * 70)
print(f'  Model:              VAE-A (WITH near-zero variance features)')
print(f'  Input dimension:    {INPUT_DIM}  (vs VAE-B: 140)')
print(f'  Latent dimension:   {QUANTUM_DIM}')
print(f'  Architecture:       {INPUT_DIM} -> 512 -> 256 -> 128 -> 8')
print(f'  Parameters:         {sum(p.numel() for p in model.parameters()):,}')
print(f'  Device:             {DEVICE}')
print(f'  Batch size:         {CONFIG["batch_size"]}')
print(f'  Epochs trained:     {len(history)}')
print(f'  Best epoch:         {checkpoint["epoch"]}')
print(f'  Best val_recon:     {checkpoint["val_recon"]:.6f}  (target: 0.0005-0.0007)')
print(f'  Best val_kl:        {checkpoint["val_kl"]:.4f}     (target: 8.0-14.0)')
print(f'  z_train shape:      {list(z_train.shape)}')
print(f'  z_test shape:       {list(z_test.shape)}')
print(f'  Angle range:        [{z_train.min():.4f}, {z_train.max():.4f}]')
print(f'  Per-dim std (mean): {z_train.std(dim=0).mean():.4f}')
print(f'  Per-dim std (min):  {z_train.std(dim=0).min():.4f}  (target: 0.60-0.95)')
print(f'  Min class distance: {validation_results["min_class_dist"]:.4f}  (target: 1.5-2.2)')
print(f'  Collapsed dims:     {validation_results["collapsed_dims"]}          (target: 0)')
print()
print('  Ablation comparison (VAE-A vs VAE-B):')
print(f'  VAE-A (this):  167 features (with near-zero variance)')
print(f'  VAE-B (done):  140 features (without near-zero variance)')
print(f'  Selection:     Compare VQC macro F1 on full test set in Stage 4')
print(f'                 Winner feeds Stage 5 ensemble')
print()
print('  Stage 4 loads:')
print(f'    {STAGE3_DIR}/vae_a_z_train.parquet')
print(f'    {STAGE3_DIR}/vae_a_z_test.parquet')
print('=' * 70)